In [ ]:
# import os
# os.environ["OPENAI_API_KEY"] = ""

In [3]:
!pip install langchain langchain-openai langchain-community chromadb tiktoken

In [5]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.prompts import ChatPromptTemplate
from langchain_community.vectorstores import Chroma
from langchain.schema import StrOutputParser, Document
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

In [26]:
# 문장 1개당 1개의 문서(문장)

# docs = [
#     Document(page_content = "RAG는 Retrieval과 Generation을 결합한 기술입니다."),
#     Document(page_content = "RAG는 외부 데이터베이스를 검색해 LLM의 답변을 보완합니다."),
#     Document(page_content = "Vector DB는 문서 임베딩을 저장해 유사 문서를 검색할 수 있습니다.")
# ]

docs = [
    Document(page_content = "최예나는 RAG수업을 듣고 있는 학생입니다."),
    Document(page_content = "최예나의 남편은 김찬영 입니다."),
    Document(page_content = "김찬영과의 결혼기념일은 8/19 입니다.")
]

In [27]:
# RAG 핵심 메서드 3가지
embedding = OpenAIEmbeddings(model='text-embedding-3-small')
db = Chroma.from_documents(docs, embedding)
retriever = db.as_retriever(search_kwargs={"k":2})

In [28]:
# 프롬프트 만들고 LCEL체인
prompt = ChatPromptTemplate.from_template("""
[RAG기반 응답]
문서내용:
{context}

질문 : {question}
""")
llm = ChatOpenAI(model = 'gpt-4o-mini', temperature = 0)# RAG 기반으로 응답을 받기 때문에 temperatur = 0
parser = StrOutputParser()

In [29]:
# LCEL로 RAG 기본 파이프라인을 구성
rag_chain = (
    {"context": retriever | RunnableLambda(lambda docs: "\n".join([d.page_content for d in docs])),
    "question" : RunnablePassthrough()}
    | prompt
    | llm
    | parser
)

In [33]:
print(rag_chain.invoke("최예나는 결혼을 했나요? 아이돌 최예나는 누구인가요"))

최예나는 결혼을 했습니다. 그녀의 남편은 김찬영입니다. 최예나는 RAG 수업을 듣고 있는 학생입니다. 아이돌 최예나는 K-pop 그룹의 멤버로 활동하는 아티스트입니다.


##PDF 파일 RAG 적용

In [34]:
!pip install -q pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 6.2 MB/s eta 0:00:00


In [35]:
from langchain_community.document_loaders import PyPDFLoader

In [36]:
pdf_path = "/content/R2510425.pdf"
loader = PyPDFLoader(pdf_path)
docs = loader.load() # pdf 파일안에 있는 내용으로 읽어오기

In [40]:
print(len(docs)) # 페이지 수

9


In [41]:
print(docs[0]) # 첫페이지 출력, 리스트 형태로 되어있

page_content='- 1 -
보도자료
보도시점 : 2025. 10. 15.(수) 10:00 이후(10. 15.(수) 석간) / 배포 : 2025. 10. 15.(수)｢주택시장 안정화 대책｣ 발표 - 규제지역 및 토지거래허가구역 서울 전역 및 경기 12개 지역 확대 지정 - 수도권ㆍ규제지역 내 15억 초과 주택 주담대 한도 제한 강화(6→2~4억원) - 과도한 부동산 투자이익이 발생하지 않도록 부동산 제도 합리화 - 부동산 거래 감독기구 설치 및 부동산 불법행위 범부처 대응역량 강화□ 국토교통부(장관 김윤덕), 기획재정부(장관 구윤철), 금융위원회(위원장 이억원), 국무조정실(실장 윤창렬), 국세청(청장 임광현)은 10.15일(수) 07:00 정부서울청사에서 ｢부동산 관계장관회의｣를 개최하여 ｢주택시장 안정화 대책｣을 논의하였다.    * (참석) 경제부총리 겸 기획재정부 장관, 국토교통부 장관, 금융위원장, 국무조정실장, 국세청장□ 참석자들은 최근 서울 및 경기도 일부 지역 중심으로 주택가격 상승세와 매매거래량 증가세가 가팔라지는 등 주택시장 불안이 확산되고 있으며, 집값 상승기대 확대에 따른 가수요 유입도 가시화되고 있어 추가적인 집값 상승 가능성을 배제하기 어렵다는 인식을 공유하였다. ㅇ 주택시장 불안은 서민 주거안정을 위협하는 문제인 동시에, 근로의욕 저하와 주거비 부담 가중에 따른 소비 위축, 자원분배 왜곡 등으로 이어져 경제 전반의 활력을 저해하는 요인인 만큼, ㅇ 주택시장 과열 양상을 조기에 차단하기 위해 선제적인 수요관리 조치를 실시하고, 가계ㆍ기업의 자본이 생산적 부문으로 투자될 수 있도록 유도해나가기로 하였다.□ 금일 발표된 ｢주택시장 안정화 대책｣의 주요 내용은 다음과 같다.  ➊ 최근 주택시장 불안이 확산되고 있는 주요 지역을 대상으로 조정대상지역, 투기과열지구 및 토지거래허가구역을 지정한다.' metadata={'producer': 'Hancom PDF 1.3.0.542', 'creator': 'Hwp 2018 10.0.0.1113

In [43]:
# chunk를 하는 메서드 RecursiceCharacterTextSplitter
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 100
)
chunks = splitter.split_documents(docs)

print(len(chunks))

15


In [52]:
print(chunks[10].page_content)
# 본문만 보는 방법은 .page_content[:300]
# 메타데이터는 .metadata

- 5 -
담당 부서국토교통부주택정책과책임자과  장이유리(044-201-3317)담당자사무관최준녕 (044-201-3318)담당자사무관이종문(044-201-3324)국토교통부토지정책과책임자과  장한정희(044-201-3398)담당자사무관이영주(044-201-3402)담당 부서기획재정부부동산정책팀책임자팀  장 최시영(044-215-2850)담당자사무관유형세(044-215-2851)기획재정부재산세제과책임자과  장이영주(044-215-4310)담당자사무관오다은(044-215-4312)담당 부서금융위원회금융정책과책임자과  장 권유이(02-2100-2830)담당자서기관윤덕기(02-2100-1690)담당자서기관이은진(02-2100-1692)


In [62]:
# persist_directory = "./pdf_vectoer_db"에 출력 되는 구문들
# chroma.sqlite3 - 메타데이터 데이터 베이스 - 문서 텍스트 , ID, 메타데이터
# SQL -> 엑셀과 비슷한 형태의 데이터
# .bin 정보를 담은 파일
# data_level0.bin - 임베딩 백터 값이 순차적으로 저장
# header - 전체 노드수, 연결수
# Length - 길이, 차원
# link_list - 다른 노드와 어떻게 연결되어있는지 저장 - 비슷한 백터간의 이웃관계

embedding = OpenAIEmbeddings(model='text-embedding-3-small')
db = Chroma.from_documents(chunks, embedding, persist_directory="./pdf_vector_db")

In [61]:
# 우리의 질문을 기반으로 임베딩에서 가장 유사한 문서를 찾아서 순서대로 출력하는 구문!
query = "주택담보대출 규제 내용은 어떤것이 있나요?"
result = db.similarity_search(query, k=2)

for i, doc in enumerate(result, 1):
  print(i)
  print(doc.page_content)

1
- 7 -
참고2 규제지역 지정기준 및 주요 효과□ 지정 기준구분조정대상지역투기과열지구정량요건➊ 공통￭ 최근 3개월간 주택가격상승률  > 물가상승률 1.3배￭ 최근 3개월간 주택가격상승률 > 물가상승률 현저히 높은곳➋ 선택*1개이상 충족필요➀ 2개월간 청약경쟁률 5:1 초과➁ 3개월간 분양권 전매거래량전년 동기 대비 30% 이상 증가➂ 주택보급률‧자가보유율전국 평균 이하➀ 2개월간 청약경쟁률 5:1 초과➁ 분양 전월대비 30% 이상 감소➂ 인허가 전년대비 급감 ➃ 주택보급률‧자가보유율 전국 평균 이하정성요건￭ 주택가격, 청약경쟁률 등 고려 시 주택분양 등 과열 또는 우려 지역￭ 지역 주택시장 여건 등 고려 시 주택 투기 성행 또는 우려 지역□ 주요 지정효과 조정대상지역투기과열지구대출▪주담대 LTV 무주택 40%, 유주택 0%, 대출한도 6억원 이하▪전세대출 1주택자 대출한도 2억원, 전세대출 보증비율 80%, 조건부 전세대출 금지▪신용대출 1억원 초과 보유차주 1년간 규제지역 내 주택 구입 제한세제▪다주택자 취득세 중과(2주 택  8% , 3주 택  12% )▪다주택자 양도소득세 중과 및 장기보유특별공제 전면 배제(한시 유예 중, '22.5~'26.5)▪양도세 1세대1주택 비과세 요건 강화-전매▪수도권 3년, 지방 1년 전매제한청약▪재당첨 제한 7년 ▪재당첨 제한 10년▪2년이상 지역 거주자 우선공급▪민영주택 가점제 적용비율 차등정비사업▪재건축 조합원당 주택 공급수 1주택으로 제한- ▪재건축·재개발 조합원 지위양도 제한기타▪자금조달계획서 및 입주계획 신고 의무▪자금조달계획서 및 입주계획 신고, 증빙자료 제출 의무
2
- 7 -
참고2 규제지역 지정기준 및 주요 효과□ 지정 기준구분조정대상지역투기과열지구정량요건➊ 공통￭ 최근 3개월간 주택가격상승률  > 물가상승률 1.3배￭ 최근 3개월간 주택가격상승률 > 물가상승률 현저히 높은곳➋ 선택*1개이상 충족필요➀ 2개월간 청약경쟁률 5:1 초과➁ 3개월간 분양권 전매거래량전년 동기 대비 30% 이상 증가➂ 주택보급

In [63]:
pdf_path = "/content/고객센터_FAQ_및_환불정책.pdf"
loader = PyPDFLoader(pdf_path)
docs = loader.load() # pdf 파일안에 있는 내용으로 읽어오기

In [64]:
print(len(docs)) # 페이지 수

2


In [75]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 100
)
chunks = splitter.split_documents(docs)

print(len(chunks))

3


In [76]:
embedding = OpenAIEmbeddings(model='text-embedding-3-small')
db = Chroma.from_documents(chunks, embedding, persist_directory="./pdf_vector_db")

In [81]:
print(chunks[2].page_content)

6️⃣ 문의 채널
📞 전화 문의: 1588-0000 💬 실시간 채팅: 홈페이지 하단 “1:1 문의하기” 버튼 클릭
📧 이메일: support@samplecorp.co.kr 🏢 주소: 서울특별시 강남구 테헤란로 123,
고객만족센터
© 2025 SampleCorp. All rights reserved.


In [84]:
query = "환불 규정은 어떻게 되나요?"
result = db.similarity_search(query, k=1)

for i, doc in enumerate(result, 1):
  print(i)
  print(doc.page_content)

1
- 8 -
참고3 주요 FAQ ※ 대출, 세제 관련 사항은 소관 부처 설명 등을 참고하시기 바랍니다. 1.규제지역 내 전매제한은 누구에게 언제부터 적용되는지?□규제지역 지정 공고일부터 즉시전매제한이 적용되나,지정일당시 분양권 기 소유자(당첨자 및 분양권 매수자)는 1회에한해 전매 허용 2.규제지역 지정에 따른 청약 규제의 적용대상·시기는?□규제지역 지정 시 청약통장 가입 기간,세대주 등 1순위 당첨에 대한 자격요건이 강화되며,가점제 적용 비율이 확대되고,청약 당첨 시 일정 기간 재당첨이 제한되는 등 제약이 발생함ㅇ 규제지역 지정 공고일이후 입주자모집승인 신청분부터규제가 적용됨3.규제지역 지정이 되면 정비사업이 받는 규제는 어떤 것들이 있고,어떻게 적용되는 것인지?□도시정비법상 규제지역 관련 규제는 조합원 지위양도 제한,재당첨제한,조합원 공급 주택수 제한등의 규정이 적용됨ㅇ 규제지역 지정공고일 당시조합설립 인가된 재건축구역 및 관리처분계획 인가된 재개발구역 부터는 조합원 지위양도가 불가    * 매매거래 자체는 가능하나 양수인은 조합원 지위를 취득할 수 없고, 현금청산 대상이 됨ㅇ 지정일부터분양 대상자로 선정된조합원및 일반 분양자는 5년내 투기과열지구에 위치한 타 정비사업에서 조합원 분양 신청 불가ㅇ 지정일부터최초로 사업시행계획 인가를 신청하는 재건축 조합은조합원당주택 공급수가 1주택으로 제한됨(1+1은 예외*)    * 종전자산의 가격 또는 주거전용면적 범위 내에서 2주택(1+1)까지 공급 가능하며,+1주택은 전용 60㎡ 이하로 하며 이전고시 후 3년간 전매제한 적용


## 새로운 벡터 DB추가



In [101]:
# 1. 기존 DB 불러오기 !
# model -> 갑자기 embadding 모델을 변경하면, 벡터에 대한 스케일이 달라지게 됨
# 각 embedding 마다 저장된 단어의 구조, embadding 계산 방식이 달라 -> 검색 정확도가 떨어질 수 있게 됩니다.
# 웬만하면 동일한 임베딩을 계속 활용하는것이 좋습니다.
embedding = OpenAIEmbeddings(model = 'text-embedding-3-small')
db_path = "./pdf_vector_db"

db = Chroma(persist_directory = db_path, embedding_function = embedding)

In [102]:
# 2. 새로운 문서 로드 및 분할해서 DB업데이트
pdf_path = "/content/2025년 종합부동산세 합산배제·특례 혜택 홈택스로 편리하게 신청하세요!.pdf"

In [103]:
# PyPDFLoader로 문서를 로드
loader = PyPDFLoader(pdf_path)
docs = loader.load()

In [104]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 100
)
chunks = splitter.split_documents(docs)

In [105]:
print(len(chunks))

35


In [124]:
# 기존 DB 객체에 새 청크를 추가
db.add_documents(chunks)
# 변경된 DB 내용을 로컬 폴더에 저장!
db.persist()

In [125]:
# 1. 기본 실습
results = db.similarity_search("주택임대 사업자 등록 방법은?" , k = 2)
for r in results:
  print(r.page_content)

- 18 -
[합산배제 관련]Q3지난해 합산배제 신고 후 변동사항이 없는 경우에도 올해 다시 신고해야 하는지?□ 변동사항이 없는 경우에는 신고하실 필요가 없습니다. ○다만, 기존에 합산배제를 신고한 과세대상 물건에 변동(소유권, 면적, 사원용주택의 임차료 등)이 있는 경우에는 신고기간 내에 물건 변동 내역을 반영하여 신고해 주시기 바랍니다.□또한, 기존 합산배제 적용 주택이 임대료 상한을 초과하는 등의 사유로 합산배제 요건을 충족하지 못하게 된 경우 합산배제 대상에서 “제외”하는 신고를 해야 합니다.Q4실제 주택을 임대하고 있음에도 임대사업자등록을 하지 못한 경우 합산배제를 적용받을 수 없는 것인지?□과세기준일(6.1.)현재 실제 주택을 임대하고 있는 납세자는○합산배제 신고기간 종료일(9.30.)까지 지자체(시·군·구 임대사업자)와 세무서(주택임대업 사업자)에 각각 등록하는 경우 공시가격 등 일정 요건 충족 시 합산배제를 적용받을 수 있습니다.Q5주택임대 사업자 등록은 방법은?□국토교통부 등록민간임대주택 렌트홈(www.renthome.go.kr)이나 주소지 관할 시·군·구청에서 본인 소유 임대주택을 일괄 등록할 수 있으며,○‘세무서 사업자등록 신청서 제출’에 동의하는 경우* 기관 간 신청정보를 주고받아 세무서 방문 없이 임대사업자 등록과 세무서 주택임대업 사업자등록을 동시에 처리할 수 있습니다.    *렌트홈 로그인 > 임대사업자 등록 신청 > ‘국세청 사업자 신고’ 체크
- 18 -
[합산배제 관련]Q3지난해 합산배제 신고 후 변동사항이 없는 경우에도 올해 다시 신고해야 하는지?□ 변동사항이 없는 경우에는 신고하실 필요가 없습니다. ○다만, 기존에 합산배제를 신고한 과세대상 물건에 변동(소유권, 면적, 사원용주택의 임차료 등)이 있는 경우에는 신고기간 내에 물건 변동 내역을 반영하여 신고해 주시기 바랍니다.□또한, 기존 합산배제 적용 주택이 임대료 상한을 초과하는 등의 사유로 합산배제 요건을 충족하지 못하게 된 경우 합산배제 대상에서 “제외”하는 신고를 해야

In [127]:
# 2. 문서 출처별 필터를 사용할수 있어요.

retriever = db.as_retriever(search_kwargs={
    "filter": {"source" : "/content/2025년 종합부동산세 합산배제·특례 혜택 홈택스로 편리하게 신청하세요!.pdf"},
    "k" : 2
})

docs = retriever.get_relevant_documents("의무임대기간은?")
print(docs[0].page_content)

- 20 -
Q10임대사업자로 등록하기 이전에 임대한 기간도 의무임대기간에 포함되는지?□임대기간은 임대사업자의 지위에서 임대한 기간을 뜻하며, ○임대사업자로 등록하기 이전의 임대기간은 합산배제 임대주택의 의무 임대기간에 포함되지 않습니다.Q11기존 등록 임대주택을 포괄양수한 경우 전 임대사업자가 임대한 기간도 의무임대기간에 포함되는지?□임대기간은 세법에 따라 납세의무자가 보유한 기간 내에서 인정되며,○등록임대주택을 포괄양수한 경우 전 임대사업자가 임대한 기간은 합산배제 임대주택의 의무임대기간에 포함되지 않습니다.Q12합산배제 임대주택을 의무임대기간 중 다른 사업자에게 포괄양도한 경우 경감받은 세액이 추징되는지?□합산배제 임대주택을 세법에서 정한 의무임대기간 중에 다른 사업자에게 포괄양도하면 그동안 경감받은 세액과 이자상당가산액이 추징됩니다.Q13합산배제 임대주택 임대료 증액제한은 언제부터 적용 되고 어떤 불이익이 있는지?□임대사업자로 등록한 이후* 최초로 체결하는 임대차계약을 기준으로 그 이후 임대차계약을 새로 체결하거나 갱신하는 때 임대료 증액 제한 위반 여부를 판단하며  * ’19.2.11. 이전에 임대사업자 등록을 한 경우에는 ’19. 2. 12. 이후○직전 임대차계약으로부터 1년 이내에 임대료 또는 임대보증금을 인상하거나, 1년이 지난 후에 인상하는 경우로서 임대료 또는 임대보증금의 증가율이 5%를 초과할 경우○과거 경감받은 세액과 이자상당가산액을 납부해야 하며, 해당연도와 그다음 연도까지 총 2년간 합산배제 대상에서 제외됩니다.


In [128]:
# 2. 문서 출처별 필터를 사용할 수 있어요.

retriever = db.as_retriever(search_kwargs = {
    "filter" : {"souce" : "/content/R2510425.pdf"},
    "k" : 2
})

docs = retriever.get_relevant_documents("의무임대기간")
print(docs[0].page_content)

IndexError: list index out of range

In [130]:
# 4. 스코어 확인 및 검색

# 각 검색결과마다의 유사도 점수!
# 0.3~0.6정도까지면 검색 효율이 좋다. (임베딩끼리의 거리가 가깝다.)
# RAG검색에서의 가장 핵심적인 지표!

docs_score = db.similarity_search_with_score("특례대상은 누구인가요?", k = 3)

for d, score in docs_score:
    print(score, d.metadata.get("source"), d.page_content[:200], "\n")

1.315098524093628 /content/2025년 종합부동산세 합산배제·특례 혜택 홈택스로 편리하게 신청하세요!.pdf - 10 -
 ○(세제혜택) 특례 신청 법인은 아래와 같이 주택분 종합부동산세 기본공제, 일반누진세율, 세부담상한을 적용받을 수 있습니다.구분기본공제세부담 상한율적용 세율2주택 이하3주택 이상특례 신청 전0원무제한2.7%(단일) 5.0%(단일)특례 신청 후9억원직전년도 대비 150%0.5% ∼ 2.7%(누진)*    *공익법인 등은 주택 사용 용도, 주택  

1.3153789043426514 /content/2025년 종합부동산세 합산배제·특례 혜택 홈택스로 편리하게 신청하세요!.pdf - 10 -
 ○(세제혜택) 특례 신청 법인은 아래와 같이 주택분 종합부동산세 기본공제, 일반누진세율, 세부담상한을 적용받을 수 있습니다.구분기본공제세부담 상한율적용 세율2주택 이하3주택 이상특례 신청 전0원무제한2.7%(단일) 5.0%(단일)특례 신청 후9억원직전년도 대비 150%0.5% ∼ 2.7%(누진)*    *공익법인 등은 주택 사용 용도, 주택  

1.3153789043426514 /content/2025년 종합부동산세 합산배제·특례 혜택 홈택스로 편리하게 신청하세요!.pdf - 10 -
 ○(세제혜택) 특례 신청 법인은 아래와 같이 주택분 종합부동산세 기본공제, 일반누진세율, 세부담상한을 적용받을 수 있습니다.구분기본공제세부담 상한율적용 세율2주택 이하3주택 이상특례 신청 전0원무제한2.7%(단일) 5.0%(단일)특례 신청 후9억원직전년도 대비 150%0.5% ∼ 2.7%(누진)*    *공익법인 등은 주택 사용 용도, 주택  



In [131]:
db = Chroma(persist_directory = db_path, embedding_function = embedding)

In [133]:
# 1. MMR 검색
# 같은 질문으로 중복된 문장이 아닌 다양한 문서를 보고싶을때 사용하는 구문

question = input("다양한 문서를 보고 싶은 질문을 입력하세요. : ")

# mmr - 유사도 + 다양성
# fetch_k 는 문서 유사한 문서 15개를 미리 뽑은 다음 중복되지 않게 k=3으로 3개를 뽑음

retr_mmr = db.as_retriever(search_type = 'mmr', search_kwargs={"k" : 3, "fetch_k": 15})
mmr_docs = retr_mmr.get_relevant_documents(question)

print(len(mmr_docs))
for i, d in enumerate(mmr_docs, 1):
  print(i, d.metadata.get("source"))
  print(d.page_content[:200], "\n")

다양한 문서를 보고 싶은 질문을 입력하세요. : 부동산 대책에 대해서 알려줘
3
1 /content/R2510425.pdf
- 1 -
보도자료
보도시점 : 2025. 10. 15.(수) 10:00 이후(10. 15.(수) 석간) / 배포 : 2025. 10. 15.(수)｢주택시장 안정화 대책｣ 발표 - 규제지역 및 토지거래허가구역 서울 전역 및 경기 12개 지역 확대 지정 - 수도권ㆍ규제지역 내 15억 초과 주택 주담대 한도 제한 강화(6→2~4억원) - 과도한 부동산 투자이익 

2 /content/2025년 종합부동산세 합산배제·특례 혜택 홈택스로 편리하게 신청하세요!.pdf
- 4 -
2 합산배제 신고 제도□대상물건 ○합산배제 신고 대상은 전용면적 및 공시가격 등 요건을 갖춘 임대주택, 사원용주택 등(기숙사, 미분양 주택 포함)과 주택건설 사업자가 주택건설을위해 취득한 토지입니다.    * 자세한 요건은 [참고2] 참조◈홈택스 ｢임대주택 합산배제 자가진단｣ 서비스를 이용하면 소유하고 있는 임대주택이 합산배제 대상인지 여부를 쉽 

3 /content/2025년 종합부동산세 합산배제·특례 혜택 홈택스로 편리하게 신청하세요!.pdf
- 13 -
참고 1 ’25년 귀속 종합부동산세 법령 주요 개정내용1.민간임대주택 종합부동산세 합산배제 대상 확대 ○30호 이상 건설 또는 30호 이상 매입해 공급하는 민간임대주택(10년 임대)에 대한 종합부동산세 합산배제 가액(공시가격) 요건 완화*    * [건설형] (현행) 9억원 이하 → (개정) 12억원 이하      [매입형] (현행) 6억원(비 



## RAG 기반 기본 챗봇 구현

In [136]:
# step1. 라이브러리 불러오기
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from langchain.prompts import ChatPromptTemplate
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser


In [147]:
# step 2. LLM모델 및 DB불러오기
embedding = OpenAIEmbeddings(model = 'text-embedding-3-small')
db = Chroma(persist_directory="./pdf_vector_db", embedding_function=embedding)
retriever = db.as_retriever(search_kwargs={"k" : 3})
llm = ChatOpenAI(model='gpt-4o-mini')

In [138]:
# step 3. 프롬프트 작성
RAG_PROMPT = ChatPromptTemplate.from_template("""
너는 부동산 정책 설명 전문가야.
아래 검색된 문서를 참고해서 질문에 한국어로 정확하게 답변해줘.
검색된 문서가 없거나 답을 모르면 '문서에 정보가 없습니다.' 라고 답변해줘.

[검색문서]
{context}

[질문] {question}
""")

In [146]:
# Step 4. LCEL 파이프라인 구성
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# 리트리터로 k=3로 이루어진 [] 리스트형식의 검색결과를 일반 텍스트로 변환
def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

rag_chain = (
    {"context" : RunnableLambda(lambda x: format_docs(retriever.invoke(x["question"]))),
     "question" : RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

In [148]:
# Step 5. 실행 테스트
query = input("질문을 입력하세요")
answer = rag_chain.invoke({"question" : query})
print(answer)

질문을 입력하세요특례에 대한 설명을 해줘
특례에 대한 설명은 다음과 같습니다. 특례 신청 법인은 주택분 종합부동산세에 대해 기본공제와 누진세율 적용, 세부담 상한 등을 받을 수 있습니다. 

- 기본공제: 특례 신청 전에는 2주택 이하의 경우 0원이 기본공제로 적용되며, 3주택 이상은 기본공제가 없습니다.
- 세부담 상한율: 특례 신청 전 2주택 이하의 경우 세부담 상한이 무제한이며, 3주택 이상은 2.7%의 단일세율이 적용됩니다. 특례 신청 후에는 9억원을 초과하는 주택에 대해 직전년도 대비 150%를 초과하는 경우 0.5%에서 2.7%의 누진세율이 적용됩니다.
- 공익법인 등에 대한 특별 규정으로, 주택 사용 용도와 수에 따라 최대 5%까지 세율이 적용될 수 있습니다. 공익법인은 직접 공익목적사업에 사용하지 않는 주택이 하나라도 있을 경우 중과누진세율이 적용될 수 있습니다.

즉, 특례를 통해 세금 부담을 경감받을 수 있는 기회를 제공받으며, 주택 수와 용도에 따라 세금 적용이 달라집니다.
